In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

In [ ]:
df = pd.read_csv('../data/nibabies_qc.tsv', sep='\t')
df = df.sort_values(by='Age months')

In [ ]:
qc_labels = {'Excellent': 1, 'Acceptable': 2, 'Poor': 3}
dot_size = 140
label_size = 14
colormap = plt.get_cmap('cividis_r')
norm = mcolors.Normalize(vmin=1, vmax=3)

def plot_reviews(ax, method):
    reviewer1 = f'{method} 1'
    reviewer2 = f'{method} 2'
    
    qc_agree = df[reviewer1] == df[reviewer2]
    colors = colormap(norm(df[qc_agree][reviewer1]))

    # Agree
    ax.scatter(
        df[qc_agree][reviewer1],
        df[qc_agree]['Age months'], 
        c=colors,
        marker='o',
        edgecolor='black',
        s=dot_size,
    )

    # Disagree
    for idx in df[~qc_agree].index:
        score_1 = df.loc[idx, reviewer1]
        score_2 = df.loc[idx, reviewer2]
        avg_score = (score_1 + score_2) / 2
        age = df.loc[idx, 'Age months']
        d_color = colormap(norm(avg_score))
        
        ax.scatter(
            avg_score,
            age,
            color=d_color,
            marker='o',
            edgecolor='black',
            s=dot_size,
        )
        ax.plot([score_1, avg_score], [age, age], color='black', linestyle=':')
        ax.plot([score_2, avg_score], [age, age], color='black', linestyle=':')

    ax.set_title(method.title(), fontsize=14)
    ax.set_xticks([1,3])
    ax.set_ylim(-1, 41)
    ax.invert_yaxis()
    ax.set_xlim(0.75, 3.25)

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(12, 10), gridspec_kw={'wspace': 0.2})

plot_reviews(axs[0], 'Surface reconstruction')
plot_reviews(axs[1], 'Spatial normalization')
plot_reviews(axs[2], 'Distortion correction')
plot_reviews(axs[3], 'Functional alignment')

fig.text(0.06, 0.52, 'Age (mo)', va='center', rotation='vertical', fontsize=14)
fig.text(0.515, 0.13, 'QC Rating', ha='center', fontsize=14)

for i, ax in enumerate(axs):
    if i > 0:
        ax.set_yticklabels([])
        
    ax.spines['top'].set_visible(False)
    ax.spines['bottom'].set_visible(False)


cbar = fig.colorbar(cm.ScalarMappable(norm=norm, cmap=colormap), ax=axs, orientation='horizontal', fraction=0.02, pad=0.06)
cbar.set_ticks(list(qc_labels.values()))
cbar.set_ticklabels(qc_labels.keys())

plt.savefig('fig4.png', bbox_inches='tight')
plt.show()